# 融合策略求解

经过融合条件判断后，若发现两个节点可以融合，可能会产生多种融合结果的组合。当存在多种融合策略时，需要确定哪种策略为最优。本节介绍融合策略最优解的求解难点、当前采用的 score + 贪心算法，以及融合顺序对结果的影响。

本节学习大纲如下：

- 融合策略最优解的求解难点
- score + 贪心融合策略求解算法
- 贪心算法的使用示例
- 当前算法的局限性


## 1. 融合策略最优解的求解难点

经过融合条件判断后，若发现两个节点可以融合，可能会产生多种融合结果的组合，例如下图中的融合策略 1 和融合策略 2。当存在多种融合策略时，需要确定哪种策略为最优。

<div style="text-align:left">
<img src="./images/fusion_strategy_options.png" alt="多种融合策略场景" width="40%">
</div>

计算整图的融合策略最优解是较为困难的，主要原因有以下两点：

1. 算法复杂度可能较高，一种可行的思路是动态规划求解，保证在 $n\log(n)$ 内完成有难度。
2. 性能估算模型不准确，没有上板实际测试前，难以准确确定融合后的真实性能。


## 2. score + 贪心融合策略求解算法

基于上述原因，追求严格的全局最优解必然涉及大量的整图融合尝试和上板实测，从而导致时间复杂度显著增加。因此，现阶段采用 score + 贪心的简单融合策略求解算法。贪心算法在每一步选择当前的局部最优结果：将图上可融合节点两两配对，逐个进行 CanFuse 判断；对于可以融合的节点对，根据 score 进行排序，以决定融合的先后顺序，最后依次进行融合处理。第一轮融合后的节点将再次重复该处理，尝试进行多轮融合；如果某一轮没有产生新的融合，流程会提前结束，默认最多融合 **10 轮**。

score 主要包括以下两项：

1. 节省的内存大小计算：利用符号实现大小比较，越大排序越靠前。
2. 节点临近性计算：使用节点对中两个节点的 topo 序 ID 差值，差值越小，排序越靠前。

排序规则如下：

首先比较节省的内存大小，内存大小相同时再比较节点临近性；若临近性也相同，则按 topo 序从小到大排列。候选节点对完成去重和排序后，后续将按所得顺序依次进行融合处理。

这里的“节点临近性”和“topo 序”作用不同：节点临近性比较的是一对节点之间的 topo 序距离，topo 序则用于在临近性相同的情况下进一步确定不同节点对的先后顺序。

## 3. 贪心算法的使用示例

以图中的 A、B、C、D 为例，它们组成 `AB`、`BC`、`CD` 融合节点对，排序后确定融合顺序。图中的 `S1`～`S3` 表示按照排序结果依次执行的三个融合处理步骤。

### 3.1 `CD` 优先，`AB` 次之

如下图所示，假设融合顺序 `CD` 优先，`AB` 次之，融合后变成 `CD`、`AB`；当处理 `BC` 融合时，B 已经变成 `AB`，C 已经变成 `CD`，最后变成 `AB + CD` 的融合处理。

<div style="text-align:left">
<img src="./images/greedy_fusion_cd_first.png" alt="CD 优先、AB 次之的融合过程" width="40%">
</div>


### 3.2 `AB` 优先，`BC` 次之

如下图所示，如果排序是 `AB` 优先，`BC` 次之，融合过程会发生变化。`AB` 先融合，在处理 `BC` 融合时，由于 B 已经变成了 `AB`，会进行 `AB + C` 的融合，融合成 `ABC`；同理，`CD` 融合会进行 `ABC + D` 的融合。

<div style="text-align:left">
<img src="./images/greedy_fusion_ab_first.png" alt="AB 优先、BC 次之的融合过程" width="40%">
</div>

两个场景的最终融合结果相同，都是 `ABCD`。然而，假设 A 和 D 不能融合，实际结果会有所不同：前一种融合顺序的结果是 `AB`、`CD`，后一种融合顺序的结果是 `ABC`、`D`。由此可见，融合顺序变化会导致融合过程和结果不同。

## 4. 当前算法的局限性

当前的贪心算法仅考虑局部最优，可能会导致全局并非最优。例如，当 A 和 D 不能融合时，如果优先融合 `CD`，可能会导致 `ABC` 无法融合，从全局角度来看，可能并非最优策略。


## 课后练习

请根据本节内容完成以下题目。

1. （判断题）经过融合条件判断后，如果存在多个融合结果组合，还需要确定融合策略。

2. （判断题）当前策略首先比较节点间的临近性，再比较节省的内存大小。

3. （判断题）当前贪心算法只考虑局部最优，不能保证整图全局最优。

4. （单选题）当前采用的融合策略求解算法是：
    A. score + 贪心算法
    B. 穷举算法
    C. 随机算法
    D. 仅使用动态规划

5. （单选题）当两个可融合节点对节省的内存大小相同时，下一步比较的是：
    A. 节点名称
    B. 节点间的临近性
    C. 模型文件大小
    D. NPU 卡号

6. （单选题）当前默认最多进行多少轮融合？
    A. 1 轮
    B. 5 轮
    C. 10 轮
    D. 64 轮

7. （多选题）计算整图融合策略最优解较为困难的原因包括：
    A. 算法复杂度可能较高
    B. 没有上板实际测试前，难以准确确定融合后的真实性能
    C. 只需按节点名称排序即可得到全局最优
    D. 图中只能存在一种融合策略

8. （多选题）关于融合顺序，以下说法正确的是哪些？
    A. 融合顺序会影响融合过程
    B. 融合顺序可能影响最终结果
    C. 处理某个候选节点对时，其中的节点可能已经属于前序步骤形成的新融合节点
    D. 不同融合顺序一定得到相同结果

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/02.03_answer.txt